---

## The Pipeline Scenario

You've been asked to investigate a critical issue with the ML pipeline.

**The ML pipeline that orchestrates your entire training and deployment workflow has failed.**

This isn't just a single model issue—the entire automated pipeline that processes data, trains models, and deploys them to production has broken down. The engineering team built this pipeline to handle deployments at scale, and now it's completely offline.

**The stakes are high:** Without this pipeline, the team can't deploy any model updates. The platform is frozen.

**Your mission:** Investigate the pipeline failure, identify the root cause, and restore the automated deployment system.

---

# Troubleshooting SageMaker Pipelines with AWS DevOps Agent

### Objective
Learn to troubleshoot SageMaker Pipeline failures by building an end-to-end ML pipeline that processes data, trains a model, registers it, and deploys it - with an intentional bug in the inference script that causes the deployment to fail.

## Workshop Overview
| Part | Step | Description |
|------|------|-------------|
| 1 | Setup | Initialize SageMaker session, create Lambda role, and generate data |
| 2 | Scripts | Create processing, training, and inference scripts (with bug) |
| 3 | Pipeline Steps | Define pipeline steps including Lambda deployment |
| 4 | Pipeline Execution | Run pipeline (will fail at deployment step) |
| 5 | Troubleshooting | Use AWS DevOps Agent to investigate the failure |
| 6 | Fix & Re-run | Apply fix and successfully deploy |

## The Bug
The inference script imports `StringIO` from the wrong location:
```python
# WRONG - deprecated/removed in newer pandas:
from pandas.io.common import StringIO

# CORRECT:
from io import StringIO
```

This causes the endpoint to fail with:
```
AttributeError: module 'pandas.io.common' has no attribute 'StringIO'
```

## Part 1: Setup

In [ ]:
!pip install -q sagemaker boto3 pandas numpy scikit-learn

In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import json
import time
import zipfile
import io
from sagemaker import get_execution_role
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.lambda_step import LambdaStep, LambdaOutput, LambdaOutputTypeEnum
from sagemaker.lambda_helper import Lambda
from sagemaker.workflow.parameters import ParameterString
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.pipeline_context import PipelineSession

# Use PipelineSession for pipeline-aware operations
pipeline_session = PipelineSession()
sagemaker_session = sagemaker.Session()
role = get_execution_role()
region = boto3.Session().region_name
bucket = sagemaker_session.default_bucket()
account_id = boto3.client('sts').get_caller_identity()['Account']
prefix = 'student-completion-pipeline'
pipeline_name = 'StudentCompletionPipeline'

print(f'Role: {role}')
print(f'Region: {region}')
print(f'Account: {account_id}')
print(f'Bucket: {bucket}')
print(f'Pipeline: {pipeline_name}')

### Create Lambda Execution Role for Pipeline Deployment

In [ ]:
# Create IAM role for Lambda deployment function
iam_client = boto3.client('iam')
lambda_role_name = 'SageMakerPipelineDeploymentLambdaRole'

lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    iam_client.create_role(
        RoleName=lambda_role_name,
        AssumeRolePolicyDocument=json.dumps(lambda_trust_policy),
        Description='Role for SageMaker Pipeline deployment Lambda'
    )
    print(f'Created role: {lambda_role_name}')
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f'Role already exists: {lambda_role_name}')

# Attach policies
policies = [
    'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
    'arn:aws:iam::aws:policy/AmazonSageMakerFullAccess'
]
for policy in policies:
    iam_client.attach_role_policy(RoleName=lambda_role_name, PolicyArn=policy)
    
lambda_role_arn = f'arn:aws:iam::{account_id}:role/{lambda_role_name}'
print(f'Lambda role ARN: {lambda_role_arn}')
time.sleep(10)  # Wait for role propagation

In [ ]:
# Generate synthetic student data
np.random.seed(42)
n_samples = 5000

df = pd.DataFrame({
    'student_id': range(1, n_samples + 1),
    'courses_enrolled': np.random.randint(1, 10, n_samples),
    'courses_completed_prev': np.random.randint(0, 8, n_samples),
    'avg_time_per_module': np.random.uniform(10, 120, n_samples),
    'login_frequency': np.random.uniform(0.5, 14, n_samples),
    'forum_posts': np.random.randint(0, 50, n_samples),
    'assignment_submission_rate': np.random.uniform(0, 1, n_samples),
    'video_completion_rate': np.random.uniform(0, 1, n_samples),
    'days_since_last_login': np.random.randint(0, 60, n_samples)
})

completion_prob = (
    0.3 * df['assignment_submission_rate'] +
    0.25 * df['video_completion_rate'] +
    np.random.normal(0, 0.1, n_samples)
)
df['completed'] = (completion_prob > 0.3).astype(int)

# Save and upload raw data
df.to_csv('raw_data.csv', index=False)
raw_data_s3 = sagemaker_session.upload_data('raw_data.csv', bucket=bucket, key_prefix=f'{prefix}/raw')
print(f'Raw data uploaded: {raw_data_s3}')
print(f'Dataset shape: {df.shape}, Completion rate: {df.completed.mean():.1%}')

In [ ]:
!mkdir -p pipeline_scripts

## Part 2: Create Pipeline Scripts

In [ ]:
%%writefile pipeline_scripts/preprocess.py
import argparse
import os
import pandas as pd
from sklearn.model_selection import train_test_split

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--input-data', type=str)
    args = parser.parse_args()
    
    input_path = '/opt/ml/processing/input/raw_data.csv'
    df = pd.read_csv(input_path)
    print(f'Loaded {len(df)} records')
    
    df = df.drop('student_id', axis=1)
    
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)
    
    train_df.to_csv('/opt/ml/processing/output/train/train.csv', index=False)
    val_df.to_csv('/opt/ml/processing/output/validation/validation.csv', index=False)
    test_df.to_csv('/opt/ml/processing/output/test/test.csv', index=False)
    
    print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
%%writefile pipeline_scripts/train.py
import argparse
import os
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--n-estimators', type=int, default=100)
    parser.add_argument('--max-depth', type=int, default=10)
    parser.add_argument('--model-dir', type=str, default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN'))
    parser.add_argument('--validation', type=str, default=os.environ.get('SM_CHANNEL_VALIDATION'))
    args = parser.parse_args()
    
    train_df = pd.read_csv(os.path.join(args.train, 'train.csv'))
    val_df = pd.read_csv(os.path.join(args.validation, 'validation.csv'))
    
    features = ['courses_enrolled', 'courses_completed_prev', 'avg_time_per_module',
                'login_frequency', 'forum_posts', 'assignment_submission_rate',
                'video_completion_rate', 'days_since_last_login']
    
    X_train, y_train = train_df[features], train_df['completed']
    X_val, y_val = val_df[features], val_df['completed']
    
    model = RandomForestClassifier(n_estimators=args.n_estimators, max_depth=args.max_depth, random_state=42)
    model.fit(X_train, y_train)
    
    val_proba = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_proba)
    acc = accuracy_score(y_val, model.predict(X_val))
    print(f'Validation AUC: {auc:.4f}, Accuracy: {acc:.4f}')
    
    joblib.dump(model, os.path.join(args.model_dir, 'model.joblib'))
    print('Model saved!')

### Inference Script (BUGGY)
This inference script has a bug - it uses the wrong import for `StringIO`.

**BUG:** The script imports `StringIO` from `pandas.io.common` instead of `io`. In newer versions of pandas, `StringIO` was removed from `pandas.io.common`, causing:
```
AttributeError: module 'pandas.io.common' has no attribute 'StringIO'
```

The endpoint will deploy successfully but fail when processing prediction requests.

In [ ]:
%%writefile pipeline_scripts/binference.py
import os
import joblib
import pandas as pd
from pandas.io.common import StringIO  # BUG: Wrong import! Should be: from io import StringIO

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, 'model.joblib'))

def input_fn(request_body, content_type):
    if content_type == 'text/csv':
        # This will fail with: AttributeError: module 'pandas.io.common' has no attribute 'StringIO'
        data = pd.read_csv(StringIO(request_body))
        return data
    raise ValueError(f'Unsupported content type: {content_type}')

def predict_fn(data, model):
    return model.predict_proba(data)[:, 1]

def output_fn(pred, accept):
    return ','.join(map(str, pred))

## Part 3: Define Pipeline Steps

In [ ]:
# Pipeline Parameters
from sagemaker.workflow.parameters import ParameterString, ParameterInteger

processing_instance_type = ParameterString(name='ProcessingInstanceType', default_value='ml.m5.large')
training_instance_type = ParameterString(name='TrainingInstanceType', default_value='ml.m5.large')
model_approval_status = ParameterString(name='ModelApprovalStatus', default_value='Approved')
endpoint_name_param = ParameterString(name='EndpointName', default_value='student-completion-pipeline-ep')

In [ ]:
# Step 1: Processing Step
sklearn_processor = SKLearnProcessor(
    framework_version='1.0-1',
    role=role,
    instance_type='ml.m5.large',
    instance_count=1,
    sagemaker_session=sagemaker_session
)

processing_step = ProcessingStep(
    name='PreprocessData',
    processor=sklearn_processor,
    inputs=[
        ProcessingInput(source=raw_data_s3, destination='/opt/ml/processing/input')
    ],
    outputs=[
        ProcessingOutput(output_name='train', source='/opt/ml/processing/output/train', destination=f's3://{bucket}/{prefix}/processed/train'),
        ProcessingOutput(output_name='validation', source='/opt/ml/processing/output/validation', destination=f's3://{bucket}/{prefix}/processed/validation'),
        ProcessingOutput(output_name='test', source='/opt/ml/processing/output/test', destination=f's3://{bucket}/{prefix}/processed/test')
    ],
    code='pipeline_scripts/preprocess.py'
)
print('Processing step defined')

In [ ]:
# Step 2: Training Step
sklearn_estimator = SKLearn(
    entry_point='train.py',
    source_dir='pipeline_scripts',
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    framework_version='1.0-1',
    py_version='py3',
    hyperparameters={'n-estimators': 100, 'max-depth': 10},
    sagemaker_session=sagemaker_session,
    base_job_name='student-pipeline-train'
)

training_step = TrainingStep(
    name='TrainModel',
    estimator=sklearn_estimator,
    inputs={
        'train': TrainingInput(s3_data=processing_step.properties.ProcessingOutputConfig.Outputs['train'].S3Output.S3Uri, content_type='text/csv'),
        'validation': TrainingInput(s3_data=processing_step.properties.ProcessingOutputConfig.Outputs['validation'].S3Output.S3Uri, content_type='text/csv')
    }
)
print('Training step defined')

In [ ]:
# Step 3: Register Model and Create Deployable Model
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.sklearn import SKLearnModel
from sagemaker.workflow.model_step import ModelStep

# Use RegisterModel for Model Registry (works with pipeline variables)
register_step = RegisterModel(
    name='RegisterModel',
    estimator=sklearn_estimator,
    model_data=training_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.m5.large', 'ml.t2.medium'],
    transform_instances=['ml.m5.large'],
    model_package_group_name='StudentCompletionModelGroup',
    approval_status='Approved'
)

# Use ModelStep with SKLearnModel for creating deployable model
# IMPORTANT: Use PipelineSession to handle pipeline variables properly
sklearn_model = SKLearnModel(
    model_data=training_step.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    entry_point='binference.py',
    source_dir='pipeline_scripts',
    framework_version='1.0-1',
    py_version='py3',
    sagemaker_session=pipeline_session  # Use PipelineSession!
)

model_step = ModelStep(
    name='CreateModel',
    step_args=sklearn_model.create(instance_type='ml.m5.large'),
    depends_on=[register_step]
)

print('Model steps defined:')
print('  1. RegisterModel - registers to Model Registry')
print('  2. CreateModel - creates deployable model with inference.py')

In [ ]:
# Step 4: Lambda Step for Deployment
# Deploys the model created by ModelStep

lambda_client = boto3.client('lambda')
lambda_function_name = 'sagemaker-pipeline-deploy-and-test'

lambda_code = '''
import boto3
import json
import time

def lambda_handler(event, context):
    print(f"Received event: {json.dumps(event)}")
    
    sm_client = boto3.client('sagemaker')
    runtime_client = boto3.client('sagemaker-runtime')
    
    model_name = event['model_name']
    endpoint_name = event.get('endpoint_name', 'student-completion-pipeline-ep')
    
    print(f"Deploying model: {model_name}")
    
    # Create endpoint config
    endpoint_config_name = f"student-ep-config-{int(time.time())}"
    sm_client.create_endpoint_config(
        EndpointConfigName=endpoint_config_name,
        ProductionVariants=[{
            'VariantName': 'AllTraffic',
            'ModelName': model_name,
            'InitialInstanceCount': 1,
            'InstanceType': 'ml.m5.large'
        }]
    )
    print(f"Created endpoint config: {endpoint_config_name}")
    
    # Delete existing endpoint if exists
    try:
        sm_client.delete_endpoint(EndpointName=endpoint_name)
        print("Deleted existing endpoint")
        time.sleep(30)
    except:
        pass
    
    # Create endpoint
    sm_client.create_endpoint(
        EndpointName=endpoint_name,
        EndpointConfigName=endpoint_config_name
    )
    print(f"Creating endpoint: {endpoint_name}")
    
    # Wait for endpoint
    waiter = sm_client.get_waiter('endpoint_in_service')
    waiter.wait(EndpointName=endpoint_name, WaiterConfig={'Delay': 30, 'MaxAttempts': 60})
    print(f"Endpoint {endpoint_name} is InService")
    
    # Test
    test_payload = """courses_enrolled,courses_completed_prev,avg_time_per_module,login_frequency,forum_posts,assignment_submission_rate,video_completion_rate,days_since_last_login
5,3,45.5,7.2,12,0.85,0.72,5"""
    
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='text/csv',
        Body=test_payload
    )
    result = response['Body'].read().decode('utf-8')
    print(f"Test prediction: {result}")
    
    return {
        'statusCode': 200,
        'endpoint_name': endpoint_name,
        'test_result': result
    }
'''

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', lambda_code)
zip_buffer.seek(0)

try:
    lambda_client.create_function(
        FunctionName=lambda_function_name,
        Runtime='python3.9',
        Role=lambda_role_arn,
        Handler='lambda_function.lambda_handler',
        Code={'ZipFile': zip_buffer.read()},
        Timeout=900,
        MemorySize=256
    )
    print(f'Created Lambda function: {lambda_function_name}')
except lambda_client.exceptions.ResourceConflictException:
    zip_buffer.seek(0)
    lambda_client.update_function_code(
        FunctionName=lambda_function_name,
        ZipFile=zip_buffer.read()
    )
    print(f'Updated Lambda function: {lambda_function_name}')

lambda_function_arn = f'arn:aws:lambda:{region}:{account_id}:function:{lambda_function_name}'

deploy_lambda = Lambda(
    function_arn=lambda_function_arn,
    execution_role_arn=lambda_role_arn,
    session=sagemaker_session
)

deploy_step = LambdaStep(
    name='DeployAndTestModel',
    lambda_func=deploy_lambda,
    inputs={
        'model_name': model_step.properties.ModelName,
        'endpoint_name': endpoint_name_param
    },
    outputs=[
        LambdaOutput(output_name='endpoint_name', output_type=LambdaOutputTypeEnum.String),
        LambdaOutput(output_name='statusCode', output_type=LambdaOutputTypeEnum.Integer)
    ]
)
print('Lambda deployment step defined')

## Part 4: Create and Execute Pipeline

In [ ]:
# Create Pipeline with all steps
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_type,
        training_instance_type,
        model_approval_status,
        endpoint_name_param
    ],
    steps=[
        processing_step,
        training_step,
        register_step,   # Register to Model Registry
        model_step,      # Create deployable model
        deploy_step      # Deploy and test
    ],
    sagemaker_session=sagemaker_session
)

pipeline.upsert(role_arn=role)
print(f'Pipeline {pipeline_name} created/updated')
print('\nPipeline flow:')
print('  Preprocess -> Train -> Register -> CreateModel -> Deploy & Test')

In [ ]:
# Execute Pipeline
execution = pipeline.start()
print(f'Pipeline execution started: {execution.arn}')
print('\nMonitor in SageMaker Studio Pipelines UI...')
print('Expected: DeployAndTestModel step will FAIL!')

In [ ]:
# Wait for pipeline completion (will fail at deploy step)
try:
    execution.wait()
except Exception as e:
    print(f'Pipeline failed as expected: {e}')
    
print(f'\nPipeline execution status: {execution.describe()["PipelineExecutionStatus"]}')

In [ ]:
# List pipeline execution steps - see which step failed
print('Pipeline Step Status:')
print('-' * 50)
for step in execution.list_steps():
    status = step['StepStatus']
    name = step['StepName']
    marker = '>>> FAILED <<<' if status == 'Failed' else ''
    print(f"{name}: {status} {marker}")
    if status == 'Failed' and 'FailureReason' in step:
        print(f"  Reason: {step['FailureReason']}")

---

## The Investigation Begins

The pipeline execution shows a clear failure at the **DeployAndTestModel** step. The Lambda function that handles deployment timed out waiting for the endpoint to become healthy.

**What you know so far:**
- ✓ Data processing completed successfully
- ✓ Model training completed successfully  
- ✓ Model registered to Model Registry
- ✓ Model created for deployment
- ✗ **Deployment Lambda timed out after 15 minutes**

The endpoint was created but never reached "InService" status. Something is preventing the container from starting properly.

**Time to use AWS DevOps Agent to dig deeper...**

---

## Part 5: Troubleshoot with AWS DevOps Agent

The pipeline failed at the **DeployAndTestModel** step. Use the AWS DevOps Agent to investigate:

**Investigation Details:**

```
The SageMaker Pipeline "StudentCompletionPipeline" failed at the DeployAndTestModel Lambda step and the model deployment of the sagemaker endpoint failed to become healthy and the Lambda timed out waiting.
```

**Investigation Starting point :**
```Please review the SageMaker pipeline in the my account in the current regions, review the the each step of the pipeline and the respective logs along with the lambda nd the sagemaker endpoint associated with the pipeline and related configurations to identify the root cause of the issue.```


**What to look for:**
1. Lambda function logs showing the waiter timeout
2. SageMaker endpoint status showing "Failed" 
3. Endpoint container logs in CloudWatch showing:
   ```
      AttributeError: module 'pandas.io.common' has no attribute 'StringIO'

   ```


**Root Cause:** The inference script imports `StringIO` from `pandas.io.common` instead of `io`. 
This import path was deprecated and removed in newer pandas versions.

---

## Root Cause Identified

After reviewing the CloudWatch logs with AWS DevOps Agent, the issue becomes clear:

```python
AttributeError: module 'pandas.io.common' has no attribute 'StringIO'
```

**The Problem:** The inference script imports `StringIO` from the wrong location. In newer versions of pandas, `StringIO` was removed from `pandas.io.common` and should be imported from Python's built-in `io` module instead.

**Why this matters:** The endpoint container starts successfully, but crashes immediately when trying to load the inference script. The Lambda function waits for the endpoint to become healthy, but it never does—leading to the timeout.

**The Fix:** Change one line in the inference script:
```python
# WRONG (deprecated):
from pandas.io.common import StringIO

# CORRECT:
from io import StringIO
```

**Time remaining:** 2 hours 15 minutes. Let's apply the fix and re-run the pipeline.

---

## Part 6: Apply Fix and Re-run Pipeline

**Root Cause Identified:** The inference script uses the wrong import for `StringIO`.

```python
# WRONG - deprecated/removed in newer pandas:
from pandas.io.common import StringIO

# CORRECT:
from io import StringIO
```

### Fix Steps
1. Write the corrected inference script to `pipeline_scripts/`
2. Recreate the `SKLearnModel` and `ModelStep` with the updated script
3. Update the `LambdaStep` to use the new model step
4. Update the pipeline and re-run
   - SageMaker step caching will automatically skip unchanged steps (`PreprocessData`, `TrainModel`, `RegisterModel`)
   - Only `CreateModel` and `DeployAndTestModel` will actually re-run

In [ ]:
# Step 1: Write the FIXED inference script with correct import 
fixed_inference_script = """import os, joblib
import pandas as pd
from io import StringIO # FIX: Correct import location

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, 'model.joblib'))

# FIX: Added ping() function for health checks
def ping():
    return ''

def input_fn(request_body, content_type):
    if content_type == 'text/csv':
        data = pd.read_csv(StringIO(request_body))
        return data
    raise ValueError(f'Unsupported content type: {content_type}')

def predict_fn(data, model):
    return model.predict_proba(data)[:, 1]

def output_fn(pred, accept):
    return ','.join(map(str, pred))
"""

with open('pipeline_scripts/inference.py', 'w') as f:
    f.write(fixed_inference_script)
print('✓ Fixed inference.py written')

# Show the diff
print('\nFix applied:')
print('  - from pandas.io.common import StringIO  # WRONG')
print('  + from io import StringIO                # CORRECT')

In [ ]:
# Step 2: Clean up broken endpoint
sm_client = boto3.client('sagemaker')
try:
    sm_client.delete_endpoint(EndpointName='student-completion-pipeline-ep')
    print('✓ Deleted broken endpoint')
    time.sleep(30)
except Exception as e:
    print(f'No endpoint to delete or error: {e}')

In [ ]:
# Step 3: Recreate ModelStep with the fixed inference script
# The SKLearnModel needs to be recreated to pick up the new inference.py

sklearn_model_fixed = SKLearnModel(
    model_data=training_step.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    entry_point='inference.py',
    source_dir='pipeline_scripts',  # Now contains the FIXED inference.py
    framework_version='1.0-1',
    py_version='py3',
    sagemaker_session=pipeline_session
)

model_step_fixed = ModelStep(
    name='CreateModel',
    step_args=sklearn_model_fixed.create(instance_type='ml.m5.large'),
    depends_on=[register_step]
)

print('✓ Created new ModelStep with fixed inference script')

In [ ]:
# Step 4: Update Lambda to use the new model step
deploy_step_fixed = LambdaStep(
    name='DeployAndTestModel',
    lambda_func=deploy_lambda,
    inputs={
        'model_name': model_step_fixed.properties.ModelName,  # Use the FIXED model step
        'endpoint_name': endpoint_name_param
    },
    outputs=[
        LambdaOutput(output_name='endpoint_name', output_type=LambdaOutputTypeEnum.String),
        LambdaOutput(output_name='statusCode', output_type=LambdaOutputTypeEnum.Integer)
    ]
)
print('✓ Updated deploy step to use fixed model')

In [ ]:
# Step 5: Update pipeline with the fixed steps and re-run
# Note: Since we changed the pipeline steps (model_step_fixed, deploy_step_fixed),
# we cannot use SelectiveExecutionConfig - it requires the same pipeline version.
# The pipeline will use step caching for unchanged steps (Preprocess, Train, Register).

pipeline_fixed = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_type,
        training_instance_type,
        model_approval_status,
        endpoint_name_param
    ],
    steps=[
        processing_step,
        training_step,
        register_step,
        model_step_fixed,    # FIXED model step
        deploy_step_fixed    # FIXED deploy step
    ],
    sagemaker_session=sagemaker_session
)

pipeline_fixed.upsert(role_arn=role)
print('✓ Pipeline updated with fixed inference script')

execution_fixed = pipeline_fixed.start(
    parameters={'EndpointName': 'student-completion-pipeline-ep-fixed'}
)
print(f'\n✓ Pipeline execution started: {execution_fixed.arn}')
print('\nPipeline will:')
print('  - Use cached results for: PreprocessData, TrainModel, RegisterModel')
print('  - Re-run: CreateModel (with fixed inference.py), DeployAndTestModel')

In [ ]:
# Step 6: Wait for pipeline completion
print('Waiting for pipeline to complete...')
print('This may take 15-20 minutes.')

try:
    execution_fixed.wait()
    print('\n✓ Pipeline completed successfully!')
except Exception as e:
    print(f'Pipeline error: {e}')

print(f'\nStatus: {execution_fixed.describe()["PipelineExecutionStatus"]}')
print('\nStep Results:')
for step in execution_fixed.list_steps():
    status = step['StepStatus']
    name = step['StepName']
    symbol = '✓' if status == 'Succeeded' else '✗'
    print(f"  {symbol} {name}: {status}")

In [ ]:
# Final Step : Test the deployed endpoint
import time
runtime_client = boto3.client('sagemaker-runtime')

# Wait a moment for endpoint to be fully ready
time.sleep(5)

test_data = """courses_enrolled,courses_completed_prev,avg_time_per_module,login_frequency,forum_posts,assignment_submission_rate,video_completion_rate,days_since_last_login
5,3,45.5,7.2,12,0.85,0.72,5
3,1,30.0,3.5,5,0.60,0.50,15
7,5,60.0,10.0,25,0.95,0.90,2"""

try:
    response = runtime_client.invoke_endpoint(
        EndpointName='student-completion-pipeline-ep-fixed',
        ContentType='text/csv',
        Body=test_data
    )
    predictions = response['Body'].read().decode('utf-8')
    print(f'✓ Endpoint working!')
    print(f'\nPredictions (completion probability):')
    for i, pred in enumerate(predictions.split(','), 1):
        print(f'  Student {i}: {float(pred):.2%}')
except Exception as e:
    print(f'Error testing endpoint: {e}')
    print('\nCheck endpoint status:')
    sm_client = boto3.client('sagemaker')
    ep = sm_client.describe_endpoint(EndpointName='student-completion-pipeline-ep-fixed')
    print(f"  Status: {ep['EndpointStatus']}")
    if ep['EndpointStatus'] == 'Failed':
        print(f"  Failure Reason: {ep.get('FailureReason', 'N/A')}")

## Summary

| Issue | Symptom | Root Cause | Fix |
|-------|---------|------------|-----|
| Wrong StringIO import | `AttributeError: module 'pandas.io.common' has no attribute 'StringIO'` | Inference script imports from deprecated `pandas.io.common` | Change to `from io import StringIO` |

### Key Learnings
1. The `StringIO` class should be imported from Python's built-in `io` module
2. `pandas.io.common.StringIO` was deprecated and removed in newer pandas versions
3. Endpoint deployment can succeed but inference fails - check CloudWatch logs for container errors
4. Always test inference scripts locally before deploying to catch import errors
5. Use `pipeline.upsert()` to upload the fixed inference script before re-running

## Pipeline Architecture

```
┌─────────────┐    ┌─────────────┐    ┌──────────────┐    ┌─────────────┐    ┌─────────────────┐
│ Preprocess  │───▶│    Train    │───▶│   Register   │───▶│ CreateModel │───▶│ Deploy & Test   │
│    Data     │    │    Model    │    │    Model     │    │             │    │                 │
└─────────────┘    └─────────────┘    └──────────────┘    └─────────────┘    └─────────────────┘
                          │                  │                   │
                          ▼                  ▼                   ▼
                    model.tar.gz      Model Registry      SageMaker Model
                                      (versioned)        (with inference.py)
```

### Bug Location
The bug is in `pipeline_scripts/inference.py` - incorrect class import `pandas.io.common.StringIO` as it is deprecated

### Files
- `pipeline_scripts/inference.py` - Inference script (BUGGY → FIXED)
- `pipeline_scripts/train.py` - Training script
- `pipeline_scripts/preprocess.py` - Data preprocessing script

In [ ]:
# Cleanup (Optional) - Run this to delete resources
# Uncomment the lines below to clean up

# sm_client = boto3.client('sagemaker')

# # Delete endpoint
# try:
#     sm_client.delete_endpoint(EndpointName='student-completion-pipeline-ep')
#     print('Deleted endpoint')
# except: pass

# # Delete endpoint configs
# for config in sm_client.list_endpoint_configs()['EndpointConfigs']:
#     if 'student-ep-config' in config['EndpointConfigName']:
#         sm_client.delete_endpoint_config(EndpointConfigName=config['EndpointConfigName'])
#         print(f"Deleted config: {config['EndpointConfigName']}")

# # Delete models
# for model in sm_client.list_models()['Models']:
#     if 'student-model' in model['ModelName'] or 'pipelines-' in model['ModelName']:
#         sm_client.delete_model(ModelName=model['ModelName'])
#         print(f"Deleted model: {model['ModelName']}")

# print('\n✓ Cleanup complete')

---

## 🎯 **Lessons Learned - The Complete Picture**

**This workshop covered troubleshooting four common ML pipeline issues:**

1. **Training Job Failure** - KeyError in training script
2. **Endpoint Health Issues** - Missing health check + memory leak
3. **Auto-scaling Configuration** - Aggressive scaling policy
4. **Pipeline Deployment Failure** - Deprecated import in inference script

### Key Takeaways

**For SageMaker Pipelines:**
- Always test inference scripts locally before deploying through pipelines
- Use step caching to speed up pipeline re-runs after fixes
- Monitor Lambda execution logs for deployment step failures
- Check CloudWatch logs for endpoint container errors

**For Production ML Systems:**
- Keep dependencies up-to-date and test for deprecated imports
- Implement comprehensive health checks for all endpoints
- Set conservative auto-scaling policies initially
- Use AWS DevOps Agent to quickly diagnose issues across services

**The Power of Automation:**
- SageMaker Pipelines orchestrate complex ML workflows
- Step caching saves time and compute costs
- Lambda steps enable custom deployment logic
- Model Registry tracks all model versions

### What You've Accomplished

You've successfully:
- ✓ Built an end-to-end ML pipeline with data processing, training, and deployment
- ✓ Debugged a production pipeline failure using AWS DevOps Agent
- ✓ Fixed a subtle import error that broke endpoint deployment
- ✓ Leveraged step caching to quickly re-deploy after fixes
- ✓ Validated the deployed model with test predictions

**You now have practical experience troubleshooting production ML operations.**

---